In [1]:
import torch
print(torch.cuda.get_device_name(0))
# Should say: NVIDIA A100-SXM4-40GB

NVIDIA A100-SXM4-40GB


In [2]:
# Check if HuggingFace token is set
import os
token = os.environ.get('HUGGINGFACE_TOKEN', None)
print(f"Token set: {token is not None}")

Token set: False


In [3]:
# ONE-TIME SETUP: Set HuggingFace token
# Run this cell once, then you never need to again this session

import os
from huggingface_hub import login

HF_TOKEN = "PASTE_YOUR_HF_TOKEN_HERE"  # paste your token here
login(token=HF_TOKEN)
os.environ['HUGGINGFACE_TOKEN'] = HF_TOKEN

print("✓ HuggingFace authenticated")

✓ HuggingFace authenticated


In [4]:
# ============================================================
# NOTEBOOK 01A: GENERAL MODELS BASELINE EVALUATION
# NS-MCA: Neuro-Symbolic Meta-Cognitive Architecture
# Author: Dedeepya Korukonda (a1945558)
# Institution: University of Adelaide
# Course: COMP 6004 | Date: May 2026
#
# Purpose: Evaluate general-purpose LLMs on MedQA-USMLE
#          to select the best foundation for NS-MCA architecture
#
# Models tested:
#   1. Flan-T5-Large  (780M) — encoder-decoder, instruction-tuned
#   2. Flan-T5-XL     (3B)   — larger encoder-decoder variant
#   3. Mistral-7B     (7B)   — strong open decoder-only model
#
# Note: Results from this notebook feed into 01C_Model_Selection
#       alongside medical model results from 01B_Medical_Models
# ============================================================

import torch
import json
import pandas as pd
import numpy as np
import time
import gc
import os
import random
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    T5ForConditionalGeneration,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig
)
from huggingface_hub import login

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

# ── HuggingFace auth ─────────────────────────────────────────
HF_TOKEN = os.environ.get('HUGGINGFACE_TOKEN', None)
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✓ HuggingFace authenticated")
else:
    print("⚠️ HF_TOKEN not set — run the auth cell first")

# ── Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_PATH = '/content/drive/My Drive/NS-MCA-Results'
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"✓ Drive mounted | Output path: {DRIVE_PATH}")

# ── GPU ──────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU: {gpu_name} | Memory: {gpu_mem:.1f} GB")
else:
    print("⚠️ No GPU — inference will be very slow")

# ── Load MedQA dataset ───────────────────────────────────────
print("\nLoading MedQA-USMLE dataset...")
with open(f'{DRIVE_PATH}/medqa_raw.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
df = pd.DataFrame(data)
print(f"✓ Loaded {len(df):,} questions")
print(f"  Columns: {list(df.columns)}")
print(f"  Splits: {df['split'].value_counts().to_dict()}")

# ── Add specialty column ──────────────────────────────────────
def extract_specialty(q):
    q = q.lower()
    if any(k in q for k in ['surgery','surgical','incision','resection',
                              'hernia','appendix','trauma','hemorrhage']):
        return 'surgery'
    if any(k in q for k in ['drug','medication','antibiotic','dose',
                              'dosage','mg','allergy','penicillin',
                              'warfarin','contraindication']):
        return 'pharmacology'
    if any(k in q for k in ['child','infant','baby','newborn',
                              'pediatric','congenital','toddler']):
        return 'pediatrics'
    return 'general'

if 'specialty_extracted' not in df.columns:
    df['specialty_extracted'] = df['question'].apply(extract_specialty)
    print("✓ Specialty column added")

print(f"  Specialty distribution:\n{df['specialty_extracted'].value_counts().to_dict()}")

print("\n" + "=" * 70)
print("✓ CELL 1 COMPLETE — Environment ready")
print("=" * 70)

✓ HuggingFace authenticated
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Drive mounted | Output path: /content/drive/My Drive/NS-MCA-Results
✓ GPU: NVIDIA A100-SXM4-40GB | Memory: 42.4 GB

Loading MedQA-USMLE dataset...
✓ Loaded 12,723 questions
  Columns: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'split']
  Splits: {'train': 10178, 'test': 1273, 'dev': 1272}
✓ Specialty column added
  Specialty distribution:
{'general': 5927, 'pharmacology': 4530, 'pediatrics': 1261, 'surgery': 1005}

✓ CELL 1 COMPLETE — Environment ready


In [5]:
# ============================================================
# ACCURACY METRICS
# ============================================================
# MedQA ground-truth 'answer' = TEXT of correct option
# e.g., "Nitrofurantoin" not "A"
#
# For generative models we cannot use classifier accuracy.
# We use two complementary metrics and report both.
# ============================================================

def metric1_answer_in_prediction(predictions, ground_truths):
    """
    PRIMARY METRIC: Does the correct answer text appear
    in the model's generated response?

    Standard approach for evaluating generative models on MC-QA.
    Conservative — requires exact string match (case-insensitive).

    Example:
      GT:   "Nitrofurantoin"
      Pred: "Nitrofurantoin is used for UTI treatment"
      → CORRECT
    """
    correct, total = 0, 0
    for pred, gt in zip(predictions, ground_truths):
        if pred is None or gt is None or str(gt).strip() == '':
            continue
        total += 1
        if str(gt).lower().strip() in str(pred).lower():
            correct += 1
    return {
        'accuracy': round((correct / total * 100), 4) if total > 0 else 0.0,
        'correct': correct,
        'total': total
    }


def metric2_option_matching(predictions, df_source):
    """
    SECONDARY METRIC: Which MC option does the prediction
    most resemble? Compare to correct answer_idx.

    Uses word-overlap scoring — robust, no arbitrary threshold.
    More rigorous than Metric 1 for short predictions.

    Example:
      Options: {A: "Nitrofurantoin", B: "Amoxicillin", ...}
      Pred: "amoxicillin is appropriate"
      Best match: B → compare to correct_idx
    """
    correct, total, skipped = 0, 0, 0
    for i, row in df_source.iterrows():
        pred = predictions[i] if i < len(predictions) else None
        if pred is None:
            continue
        options     = row.get('options', {})
        correct_idx = row.get('answer_idx', '')
        if not options or not correct_idx:
            skipped += 1
            continue
        total += 1
        pred_lower  = str(pred).lower()
        best_idx, best_score = None, -1
        for opt_key, opt_text in options.items():
            score = len(
                set(str(opt_text).lower().split()) &
                set(pred_lower.split())
            )
            if score > best_score:
                best_score = score
                best_idx   = opt_key
        if best_idx == correct_idx:
            correct += 1
    return {
        'accuracy': round((correct / total * 100), 4) if total > 0 else 0.0,
        'correct': correct,
        'total': total,
        'skipped': skipped
    }


def specialty_breakdown(predictions, ground_truths, specialties):
    """Metric 1 accuracy broken down by medical specialty."""
    from collections import defaultdict
    results = defaultdict(lambda: {'correct': 0, 'total': 0})
    for pred, gt, spec in zip(predictions, ground_truths, specialties):
        if pred is None or gt is None:
            continue
        results[spec]['total'] += 1
        if str(gt).lower().strip() in str(pred).lower():
            results[spec]['correct'] += 1
    return {
        spec: {
            'accuracy': round(
                (v['correct'] / v['total'] * 100), 4
            ) if v['total'] > 0 else 0.0,
            'correct':  v['correct'],
            'total':    v['total']
        }
        for spec, v in results.items()
    }


def print_results(model_name, m1, m2, spec, runtime, errors=0):
    """Standardised results printer for all models."""
    print(f"\n{'=' * 70}")
    print(f"RESULTS: {model_name}")
    print(f"{'=' * 70}")
    print(f"\nMetric 1 — Answer-in-Prediction (PRIMARY):")
    print(f"  Accuracy : {m1['accuracy']:.2f}%")
    print(f"  Correct  : {m1['correct']:,} / {m1['total']:,}")
    print(f"\nMetric 2 — Option Matching (SECONDARY):")
    print(f"  Accuracy : {m2['accuracy']:.2f}%")
    print(f"  Correct  : {m2['correct']:,} / {m2['total']:,}")
    print(f"\nAccuracy by Specialty:")
    for s, r in sorted(spec.items()):
        print(f"  {s:<15}: {r['accuracy']:.2f}%  ({r['correct']}/{r['total']})")
    print(f"\nRuntime  : {runtime:.1f} min")
    print(f"Errors   : {errors}")


def save_model_results(model_key, model_name, predictions,
                       m1, m2, spec, runtime, errors,
                       extra_meta=None):
    """Save predictions and metrics to Google Drive."""
    output = {
        'model_key':    model_key,
        'model_name':   model_name,
        'total':        len(predictions),
        'errors':       errors,
        'runtime_min':  round(runtime, 2),
        'metrics': {
            'metric1_answer_in_prediction': m1,
            'metric2_option_matching':      m2,
            'specialty_breakdown':          spec
        },
        'predictions': predictions,
        'extra_meta':  extra_meta or {}
    }
    path = f'{DRIVE_PATH}/{model_key}_results.json'
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2)
    print(f"✓ Saved: {model_key}_results.json")
    return path


print("✓ CELL 2 COMPLETE — Metrics defined")

✓ CELL 2 COMPLETE — Metrics defined


In [6]:
# ============================================================
# MODEL 1: FLAN-T5-LARGE (780M)
# Pre-computed in Notebook 01 — loading saved predictions
# No inference needed. Runtime: ~0 min
# ============================================================

print("=" * 70)
print("MODEL 1: FLAN-T5-LARGE (780M)")
print("Source: Notebook 01 pre-computed predictions")
print("=" * 70)

with open(f'{DRIVE_PATH}/layer1_inference_outputs.json',
          'r', encoding='utf-8') as f:
    layer1_outputs = json.load(f)

preds_list = layer1_outputs['predictions']
print(f"✓ Loaded {len(preds_list):,} predictions")
print(f"  Model     : {layer1_outputs['metadata']['model']}")
print(f"  Timestamp : {layer1_outputs['metadata']['timestamp']}")

flan_large_preds = [item.get('predicted_answer', None) for item in preds_list]
flan_large_gts   = [item.get('ground_truth_answer', None) for item in preds_list]
flan_large_confs = [item.get('confidence', 0.0) for item in preds_list]

# Metrics
fl_m1   = metric1_answer_in_prediction(flan_large_preds, flan_large_gts)
fl_m2   = metric2_option_matching(flan_large_preds, df.reset_index(drop=True))
fl_spec = specialty_breakdown(
    flan_large_preds, flan_large_gts,
    df['specialty_extracted'].tolist()
)

print(f"\nConfidence scores (proper log-probability from Notebook 01):")
print(f"  Mean : {np.mean(flan_large_confs):.4f}")
print(f"  Std  : {np.std(flan_large_confs):.4f}")
print(f"  Min  : {np.min(flan_large_confs):.6f}")
print(f"  Max  : {np.max(flan_large_confs):.4f}")

FLAN_LARGE_RUNTIME = 73.2  # confirmed from Notebook 01 actual run on T4

print_results("Flan-T5-Large", fl_m1, fl_m2, fl_spec, FLAN_LARGE_RUNTIME)

# Sample predictions
print(f"\nSample predictions:")
for i in [0, 500, 1000, 5000, 10000]:
    if i < len(flan_large_preds):
        gt   = flan_large_gts[i]
        pred = flan_large_preds[i]
        match = '✓' if gt and pred and str(gt).lower() in str(pred).lower() else '✗'
        print(f"  [{i}] {match}  GT: {str(gt)[:40]:<42} Pred: {str(pred)[:50] if pred else 'None'}")

save_model_results(
    model_key   = 'flan_t5_large',
    model_name  = 'google/flan-t5-large',
    predictions = flan_large_preds,
    m1          = fl_m1,
    m2          = fl_m2,
    spec        = fl_spec,
    runtime     = FLAN_LARGE_RUNTIME,
    errors      = 0,
    extra_meta  = {
        'source':            'Notebook 01 pre-computed',
        'confidence_method': 'token_log_probability',
        'confidence_mean':   float(np.mean(flan_large_confs)),
        'confidence_std':    float(np.std(flan_large_confs)),
        'parameters':        '780M',
        'architecture':      'encoder-decoder'
    }
)

print("\n✓ CELL 3 COMPLETE — Flan-T5-Large evaluated")

MODEL 1: FLAN-T5-LARGE (780M)
Source: Notebook 01 pre-computed predictions
✓ Loaded 12,723 predictions
  Model     : google/flan-t5-large
  Timestamp : 2026-05-17T04:14:52.202514

Confidence scores (proper log-probability from Notebook 01):
  Mean : 0.0697
  Std  : 0.1463
  Min  : 0.000002
  Max  : 0.9928

RESULTS: Flan-T5-Large

Metric 1 — Answer-in-Prediction (PRIMARY):
  Accuracy : 0.59%
  Correct  : 75 / 12,723

Metric 2 — Option Matching (SECONDARY):
  Accuracy : 19.85%
  Correct  : 2,526 / 12,723

Accuracy by Specialty:
  general        : 0.64%  (38/5927)
  pediatrics     : 0.56%  (7/1261)
  pharmacology   : 0.57%  (26/4530)
  surgery        : 0.40%  (4/1005)

Runtime  : 73.2 min
Errors   : 0

Sample predictions:
  [0] ✗  GT: Nitrofurantoin                             Pred: Pregnancy
  [500] ✗  GT: Duodenal atresia                           Pred: pyloric stenosis
  [1000] ✗  GT: Membranous nephropathy                     Pred: Adenocarcinoma
  [5000] ✗  GT: Neisseria meningitidis

In [7]:
# ============================================================
# MODEL 2: FLAN-T5-XL (3B)
# Larger variant of Flan-T5 — tests if scale helps
# Expected runtime: ~25-30 min on A100
# ============================================================

print("=" * 70)
print("MODEL 2: FLAN-T5-XL (3B)")
print("Expected runtime: ~25-30 min on A100")
print("=" * 70)

MODEL_NAME = 'google/flan-t5-xl'
xl_preds   = []
xl_errors  = 0

try:
    print(f"\nLoading {MODEL_NAME}...")
    xl_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    xl_model     = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        torch_dtype  = torch.float16,
        device_map   = 'auto',
        token        = HF_TOKEN
    )
    xl_model.eval()
    print(f"✓ Flan-T5-XL loaded")
    print(f"  Parameters: ~3B")

    xl_start = time.time()

    for i, row in enumerate(df.itertuples(index=False)):

        # Progress + partial save every 1000
        if i % 1000 == 0:
            elapsed = time.time() - xl_start
            rate    = i / elapsed if elapsed > 0 and i > 0 else 0
            eta     = (len(df) - i) / rate / 60 if rate > 0 else 0
            print(f"  {i:,}/{len(df):,} | "
                  f"Elapsed: {elapsed/60:.1f}m | ETA: {eta:.1f}m")
            if i > 0:
                with open(f'{DRIVE_PATH}/flan_t5_xl_partial_{i}.json', 'w') as f:
                    json.dump({'completed': i, 'predictions': xl_preds}, f)

        try:
            prompt = (
                f"You are a medical expert. "
                f"Answer the following clinical question with "
                f"the most appropriate answer only.\n\n"
                f"Question: {row.question}\n\n"
                f"Answer:"
            )
            inputs = xl_tokenizer(
                prompt,
                return_tensors = 'pt',
                max_length     = 512,
                truncation     = True
            ).to(device)

            with torch.no_grad():
                outputs = xl_model.generate(
                    inputs['input_ids'],
                    max_new_tokens  = 64,
                    num_beams       = 4,
                    early_stopping  = True,
                    no_repeat_ngram_size = 3
                )

            pred = xl_tokenizer.decode(
                outputs[0], skip_special_tokens=True
            ).strip()
            xl_preds.append(pred)

        except Exception as e:
            xl_preds.append(None)
            xl_errors += 1

    xl_elapsed = time.time() - xl_start
    print(f"\n✓ Inference complete: {xl_elapsed/60:.1f} min | Errors: {xl_errors}")

except Exception as e:
    print(f"❌ Failed to load Flan-T5-XL: {e}")
    xl_elapsed = 0
    xl_errors  = len(df)

finally:
    # Always clean up GPU memory
    try:
        del xl_model
        del xl_tokenizer
    except:
        pass
    torch.cuda.empty_cache()
    gc.collect()
    print("✓ GPU memory cleared")

# Metrics
xl_gts  = df['answer'].tolist()
xl_m1   = metric1_answer_in_prediction(xl_preds, xl_gts)
xl_m2   = metric2_option_matching(xl_preds, df.reset_index(drop=True))
xl_spec = specialty_breakdown(
    xl_preds, xl_gts, df['specialty_extracted'].tolist()
)

print_results("Flan-T5-XL", xl_m1, xl_m2, xl_spec,
              xl_elapsed / 60, xl_errors)

# Sample predictions
print(f"\nSample predictions:")
for i in [0, 500, 1000, 5000, 10000]:
    if i < len(xl_preds):
        gt   = xl_gts[i]
        pred = xl_preds[i]
        match = '✓' if gt and pred and str(gt).lower() in str(pred).lower() else '✗'
        print(f"  [{i}] {match}  GT: {str(gt)[:40]:<42} "
              f"Pred: {str(pred)[:50] if pred else 'None'}")

save_model_results(
    model_key   = 'flan_t5_xl',
    model_name  = MODEL_NAME,
    predictions = xl_preds,
    m1          = xl_m1,
    m2          = xl_m2,
    spec        = xl_spec,
    runtime     = xl_elapsed / 60,
    errors      = xl_errors,
    extra_meta  = {
        'parameters':   '3B',
        'architecture': 'encoder-decoder',
        'prompt_style': 'instruction'
    }
)

print("\n✓ CELL 4 COMPLETE — Flan-T5-XL evaluated")

MODEL 2: FLAN-T5-XL (3B)
Expected runtime: ~25-30 min on A100

Loading google/flan-t5-xl...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✓ Flan-T5-XL loaded
  Parameters: ~3B
  0/12,723 | Elapsed: 0.0m | ETA: 0.0m
  1,000/12,723 | Elapsed: 8.1m | ETA: 95.2m
  2,000/12,723 | Elapsed: 15.9m | ETA: 85.4m
  3,000/12,723 | Elapsed: 24.0m | ETA: 77.9m
  4,000/12,723 | Elapsed: 31.9m | ETA: 69.6m
  5,000/12,723 | Elapsed: 39.9m | ETA: 61.7m
  6,000/12,723 | Elapsed: 47.9m | ETA: 53.6m
  7,000/12,723 | Elapsed: 56.0m | ETA: 45.8m
  8,000/12,723 | Elapsed: 64.1m | ETA: 37.9m
  9,000/12,723 | Elapsed: 72.2m | ETA: 29.9m
  10,000/12,723 | Elapsed: 80.2m | ETA: 21.8m
  11,000/12,723 | Elapsed: 88.2m | ETA: 13.8m
  12,000/12,723 | Elapsed: 96.5m | ETA: 5.8m

✓ Inference complete: 102.4 min | Errors: 0
✓ GPU memory cleared

RESULTS: Flan-T5-XL

Metric 1 — Answer-in-Prediction (PRIMARY):
  Accuracy : 0.97%
  Correct  : 124 / 12,723

Metric 2 — Option Matching (SECONDARY):
  Accuracy : 20.55%
  Correct  : 2,614 / 12,723

Accuracy by Specialty:
  general        : 1.05%  (62/5927)
  pediatrics     : 1.27%  (16/1261)
  pharmacology   : 0.

In [7]:
# ============================================================
# MODEL 3: MISTRAL-7B-v0.1 (7B) — FRESH SESSION CELL
# Must run in a fresh runtime after restart
# ============================================================

# Step 1: Install BEFORE any imports
import subprocess
result = subprocess.run(
    ['pip', 'install', '-q', '-U', 'bitsandbytes>=0.46.1', 'accelerate>=0.26.0'],
    capture_output=True, text=True
)
print("Install output:", result.stdout[-200:] if result.stdout else "done")
print("Install errors:", result.stderr[-200:] if result.stderr else "none")

# Step 2: Verify version
import bitsandbytes as bnb
print(f"✓ bitsandbytes version: {bnb.__version__}")
# Must show >= 0.46.1

# Step 3: Now import everything else
import torch
import json
import pandas as pd
import numpy as np
import time
import gc
import os
import random
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from huggingface_hub import login

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Auth
HF_TOKEN = "PASTE_YOUR_HF_TOKEN_HERE" #Paste your hf token
login(token=HF_TOKEN)
os.environ['HUGGINGFACE_TOKEN'] = HF_TOKEN
print("✓ HuggingFace authenticated")

# Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_PATH = '/content/drive/My Drive/NS-MCA-Results'
print(f"✓ Drive mounted")

# GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

# Dataset
with open(f'{DRIVE_PATH}/medqa_raw.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
df = pd.DataFrame(data)
print(f"✓ Dataset: {len(df):,} questions")

def extract_specialty(q):
    q = q.lower()
    if any(k in q for k in ['surgery','surgical','incision','resection','hernia']):
        return 'surgery'
    if any(k in q for k in ['drug','medication','antibiotic','dose','mg','allergy','penicillin','warfarin']):
        return 'pharmacology'
    if any(k in q for k in ['child','infant','baby','newborn','pediatric','congenital']):
        return 'pediatrics'
    return 'general'

if 'specialty_extracted' not in df.columns:
    df['specialty_extracted'] = df['question'].apply(extract_specialty)

# Metric functions (copied here so cell is self-contained)
def metric1_answer_in_prediction(predictions, ground_truths):
    correct, total = 0, 0
    for pred, gt in zip(predictions, ground_truths):
        if pred is None or gt is None or str(gt).strip() == '':
            continue
        total += 1
        if str(gt).lower().strip() in str(pred).lower():
            correct += 1
    return {
        'accuracy': round((correct / total * 100), 4) if total > 0 else 0.0,
        'correct': correct, 'total': total
    }

def metric2_option_matching(predictions, df_source):
    correct, total, skipped = 0, 0, 0
    for i, row in df_source.iterrows():
        pred = predictions[i] if i < len(predictions) else None
        if pred is None:
            continue
        options     = row.get('options', {})
        correct_idx = row.get('answer_idx', '')
        if not options or not correct_idx:
            skipped += 1
            continue
        total += 1
        pred_lower  = str(pred).lower()
        best_idx, best_score = None, -1
        for opt_key, opt_text in options.items():
            score = len(
                set(str(opt_text).lower().split()) &
                set(pred_lower.split())
            )
            if score > best_score:
                best_score = score
                best_idx   = opt_key
        if best_idx == correct_idx:
            correct += 1
    return {
        'accuracy': round((correct / total * 100), 4) if total > 0 else 0.0,
        'correct': correct, 'total': total
    }

def specialty_breakdown(predictions, ground_truths, specialties):
    from collections import defaultdict
    results = defaultdict(lambda: {'correct': 0, 'total': 0})
    for pred, gt, spec in zip(predictions, ground_truths, specialties):
        if pred is None or gt is None:
            continue
        results[spec]['total'] += 1
        if str(gt).lower().strip() in str(pred).lower():
            results[spec]['correct'] += 1
    return {
        spec: {
            'accuracy': round((v['correct']/v['total']*100), 4) if v['total'] > 0 else 0.0,
            'correct': v['correct'], 'total': v['total']
        }
        for spec, v in results.items()
    }

def save_model_results(model_key, model_name, predictions,
                       m1, m2, spec, runtime, errors, extra_meta=None):
    output = {
        'model_key': model_key, 'model_name': model_name,
        'total': len(predictions), 'errors': errors,
        'runtime_min': round(runtime, 2),
        'metrics': {
            'metric1_answer_in_prediction': m1,
            'metric2_option_matching': m2,
            'specialty_breakdown': spec
        },
        'predictions': predictions,
        'extra_meta': extra_meta or {}
    }
    path = f'{DRIVE_PATH}/{model_key}_results.json'
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2)
    print(f"✓ Saved: {model_key}_results.json")

print("\n✓ SETUP COMPLETE — Starting Mistral-7B inference")
print("=" * 70)

# ── MISTRAL INFERENCE ─────────────────────────────────────────────────
MODEL_NAME     = 'mistralai/Mistral-7B-v0.1'
mistral_preds  = []
mistral_errors = 0
mistral_elapsed = 0

try:
    print(f"\nLoading {MODEL_NAME} with 4-bit quantization...")

    quant_config = BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_compute_dtype    = torch.float16,
        bnb_4bit_quant_type       = 'nf4',
        bnb_4bit_use_double_quant = True
    )

    mistral_tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, token=HF_TOKEN
    )
    if mistral_tokenizer.pad_token is None:
        mistral_tokenizer.pad_token = mistral_tokenizer.eos_token

    mistral_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config = quant_config,
        device_map          = 'auto',
        token               = HF_TOKEN
    )
    mistral_model.eval()
    print(f"✓ Mistral-7B loaded successfully")

    mistral_start = time.time()

    for i, row in enumerate(df.itertuples(index=False)):
        if i % 1000 == 0:
            elapsed = time.time() - mistral_start
            rate    = i / elapsed if elapsed > 0 and i > 0 else 0
            eta     = (len(df) - i) / rate / 60 if rate > 0 else 0
            print(f"  {i:,}/{len(df):,} | "
                  f"Elapsed: {elapsed/60:.1f}m | ETA: {eta:.1f}m")
            if i > 0:
                with open(f'{DRIVE_PATH}/mistral_partial_{i}.json', 'w') as f:
                    json.dump({'completed': i, 'predictions': mistral_preds}, f)

        try:
            prompt = (
                f"[INST] You are a medical expert. "
                f"Answer the following clinical question "
                f"with the correct answer only — no explanation.\n\n"
                f"Question: {row.question} [/INST]"
            )
            inputs = mistral_tokenizer(
                prompt,
                return_tensors = 'pt',
                max_length     = 512,
                truncation     = True
            ).to(device)

            with torch.no_grad():
                outputs = mistral_model.generate(
                    inputs['input_ids'],
                    max_new_tokens = 50,
                    do_sample      = False,
                    pad_token_id   = mistral_tokenizer.eos_token_id
                )

            new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
            pred = mistral_tokenizer.decode(
                new_tokens, skip_special_tokens=True
            ).strip()
            mistral_preds.append(pred)

        except Exception as e:
            mistral_preds.append(None)
            mistral_errors += 1

    mistral_elapsed = time.time() - mistral_start
    print(f"\n✓ Done: {mistral_elapsed/60:.1f} min | Errors: {mistral_errors}")

except Exception as e:
    print(f"❌ Failed: {e}")
    mistral_errors = len(df)

finally:
    try:
        del mistral_model
        del mistral_tokenizer
    except:
        pass
    torch.cuda.empty_cache()
    gc.collect()
    print("✓ GPU memory cleared")

# ── METRICS ───────────────────────────────────────────────────────────
mistral_gts  = df['answer'].tolist()
mistral_m1   = metric1_answer_in_prediction(mistral_preds, mistral_gts)
mistral_m2   = metric2_option_matching(mistral_preds, df.reset_index(drop=True))
mistral_spec = specialty_breakdown(
    mistral_preds, mistral_gts,
    df['specialty_extracted'].tolist()
)

print(f"\n{'=' * 70}")
print(f"RESULTS: Mistral-7B")
print(f"{'=' * 70}")
print(f"\nMetric 1 — Answer-in-Prediction (PRIMARY):")
print(f"  Accuracy : {mistral_m1['accuracy']:.2f}%  "
      f"({mistral_m1['correct']:,}/{mistral_m1['total']:,})")
print(f"\nMetric 2 — Option Matching (SECONDARY):")
print(f"  Accuracy : {mistral_m2['accuracy']:.2f}%  "
      f"({mistral_m2['correct']:,}/{mistral_m2['total']:,})")
print(f"\nAccuracy by Specialty:")
for s, r in sorted(mistral_spec.items()):
    print(f"  {s:<15}: {r['accuracy']:.2f}%  ({r['correct']}/{r['total']})")
print(f"\nRuntime  : {mistral_elapsed/60:.1f} min")
print(f"Errors   : {mistral_errors}")

print(f"\nSample predictions:")
for i in [0, 500, 1000, 5000, 10000]:
    if i < len(mistral_preds):
        gt    = mistral_gts[i]
        pred  = mistral_preds[i]
        match = '✓' if gt and pred and str(gt).lower() in str(pred).lower() else '✗'
        print(f"  [{i}] {match}  GT: {str(gt)[:40]:<42} "
              f"Pred: {str(pred)[:50] if pred else 'None'}")

save_model_results(
    model_key   = 'mistral_7b',
    model_name  = MODEL_NAME,
    predictions = mistral_preds,
    m1          = mistral_m1,
    m2          = mistral_m2,
    spec        = mistral_spec,
    runtime     = mistral_elapsed / 60,
    errors      = mistral_errors,
    extra_meta  = {
        'parameters':   '7B',
        'architecture': 'decoder-only',
        'quantization': '4-bit NF4',
        'prompt_style': 'mistral-instruct'
    }
)

print("\n✓ MISTRAL-7B COMPLETE")

Install output: done
Install errors: none
✓ bitsandbytes version: 0.49.2
✓ HuggingFace authenticated
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Drive mounted
✓ GPU: NVIDIA A100-SXM4-40GB
✓ Dataset: 12,723 questions

✓ SETUP COMPLETE — Starting Mistral-7B inference

Loading mistralai/Mistral-7B-v0.1 with 4-bit quantization...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


✓ Mistral-7B loaded successfully
  0/12,723 | Elapsed: 0.0m | ETA: 0.0m
  1,000/12,723 | Elapsed: 66.8m | ETA: 782.6m
  2,000/12,723 | Elapsed: 133.5m | ETA: 715.6m
  3,000/12,723 | Elapsed: 199.9m | ETA: 647.9m
  4,000/12,723 | Elapsed: 267.0m | ETA: 582.3m
  5,000/12,723 | Elapsed: 334.3m | ETA: 516.4m
  6,000/12,723 | Elapsed: 401.4m | ETA: 449.8m
  7,000/12,723 | Elapsed: 468.3m | ETA: 382.9m
  8,000/12,723 | Elapsed: 535.2m | ETA: 316.0m
  9,000/12,723 | Elapsed: 601.9m | ETA: 249.0m
  10,000/12,723 | Elapsed: 668.6m | ETA: 182.1m
  11,000/12,723 | Elapsed: 735.6m | ETA: 115.2m
  12,000/12,723 | Elapsed: 802.8m | ETA: 48.4m

✓ Done: 851.2 min | Errors: 0
✓ GPU memory cleared

RESULTS: Mistral-7B

Metric 1 — Answer-in-Prediction (PRIMARY):
  Accuracy : 9.88%  (1,257/12,723)

Metric 2 — Option Matching (SECONDARY):
  Accuracy : 24.81%  (3,157/12,723)

Accuracy by Specialty:
  general        : 10.23%  (631/6169)
  pediatrics     : 9.83%  (127/1292)
  pharmacology   : 9.36%  (435/4648

In [9]:
# ============================================================
# FIX: Update 01A summary with correct Mistral results
# Run this after Mistral completes
# ============================================================

import json
import pandas as pd

DRIVE_PATH = '/content/drive/My Drive/NS-MCA-Results'

# Load existing summary
with open(f'{DRIVE_PATH}/01A_general_models_summary.json', 'r') as f:
    summary = json.load(f)

# Update Mistral with real results
summary['models']['mistral_7b'] = {
    'model_name':      'mistralai/Mistral-7B-v0.1',
    'parameters':      '7B',
    'architecture':    'decoder-only',
    'metric1':         9.88,
    'metric2':         24.81,
    'runtime_min':     851.2,
    'errors':          0,
    'confidence_method': 'not_computed',
    'quantization':    '4-bit NF4',
    'prompt_style':    'mistral-instruct',
    'note':            (
        'Mistral generated structured option lists due to instruct '
        'prompt format — [INST] caused verbose MC-formatted output. '
        'Metric 1 partially inflated by answer text appearing in '
        'verbose output. True accuracy likely closer to Metric 2.'
    ),
    'specialty': {
        'general':      {'accuracy': 10.23, 'correct': 631,  'total': 6169},
        'pediatrics':   {'accuracy': 9.83,  'correct': 127,  'total': 1292},
        'pharmacology': {'accuracy': 9.36,  'correct': 435,  'total': 4648},
        'surgery':      {'accuracy': 10.42, 'correct': 64,   'total': 614}
    }
}

summary['date_updated'] = str(pd.Timestamp.now())

# Save updated summary
with open(f'{DRIVE_PATH}/01A_general_models_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("✓ 01A summary updated with correct Mistral results")

# Print final comparison
print(f"\n{'Model':<20} {'Params':<8} {'Metric1%':<12} {'Metric2%':<12} {'Runtime(min)'}")
print("-" * 65)
for key, m in summary['models'].items():
    print(f"{m['model_name'].split('/')[-1]:<20} "
          f"{m['parameters']:<8} "
          f"{m['metric1']:<12.2f} "
          f"{m['metric2']:<12.2f} "
          f"{m['runtime_min']:.1f}")

print(f"\nKey finding: All general LLMs perform at or near chance")
print(f"on MedQA-USMLE without medical domain fine-tuning.")
print(f"This validates the need for NS-MCA safety architecture.")
print(f"\n✓ Ready for 01B_Medical_Models.ipynb")

✓ 01A summary updated with correct Mistral results

Model                Params   Metric1%     Metric2%     Runtime(min)
-----------------------------------------------------------------
flan-t5-large        780M     0.59         19.85        73.2
flan-t5-xl           3B       0.97         20.55        102.4
Mistral-7B-v0.1      7B       9.88         24.81        851.2

Key finding: All general LLMs perform at or near chance
on MedQA-USMLE without medical domain fine-tuning.
This validates the need for NS-MCA safety architecture.

✓ Ready for 01B_Medical_Models.ipynb


In [10]:
# Keep-alive cell — do not run, just keep selected